In [1]:
import os
import joblib
import faiss
import numpy as np

# -----------------------------
# Load Category Model
# -----------------------------
category_model = joblib.load("../models/category_svm.pkl")
category_tfidf = joblib.load("../models/category_tfidf.pkl")

# -----------------------------
# Load Priority Model
# -----------------------------
priority_model = joblib.load("../models/priority/priority_model.pkl")
priority_tfidf = joblib.load("../models/priority/tfidf_vectorizer.pkl")
category_encoder = joblib.load("../models/priority/category_encoder.pkl")

# -----------------------------
# Load FAISS Index
# -----------------------------
index = faiss.read_index("../models/embeddings/ticket_index.faiss")

# -----------------------------
# Load Ticket Embeddings
# -----------------------------
ticket_embeddings = np.load("../models/embeddings/ticket_embeddings.npy")

print("✅ All models and files loaded successfully!")

print("\nCategory model:", type(category_model))
print("Priority model:", type(priority_model))
print("FAISS index:", index.ntotal)
print("Embeddings shape:", ticket_embeddings.shape)

✅ All models and files loaded successfully!

Category model: <class 'sklearn.svm._classes.LinearSVC'>
Priority model: <class 'sklearn.svm._classes.LinearSVC'>
FAISS index: 20000
Embeddings shape: (20000, 384)


In [2]:
# -----------------------------
# Category Prediction
# -----------------------------
category_text = category_tfidf.transform([ticket])
predicted_category = category_model.predict(category_text)[0]

# -----------------------------
# Priority Prediction
# -----------------------------
priority_text = priority_tfidf.transform([ticket])

# Encode predicted category
category_encoded = category_encoder.transform([[predicted_category]])

# Combine TF-IDF + encoded category
from scipy.sparse import hstack

priority_input = hstack([
    priority_text,
    category_encoded
])

predicted_priority_encoded = priority_model.predict(priority_input)[0]

print("Ticket:")
print(ticket)

print("\nPredicted Category:", predicted_category)
print("Predicted Priority:", predicted_priority_encoded)

NameError: name 'ticket' is not defined

In [ ]:
print("Encoder expects:", category_encoder.n_features_in_)
print("Encoder categories:")
print(category_encoder.categories_)

Encoder expects: 2
Encoder categories:
[array(['Billing and Payments', 'Customer Service', 'General Inquiry',
       'Human Resources', 'IT Support', 'Product Support',
       'Returns and Exchanges', 'Sales and Pre-Sales',
       'Service Outages and Maintenance', 'Technical Support'],
      dtype=object), array(['Change', 'Incident', 'Problem', 'Request'], dtype=object)]


In [ ]:
print("Priority model features:", priority_model.n_features_in_)
print("Encoder categories:", category_encoder.categories_)

Priority model features: 30014
Encoder categories: [array(['Billing and Payments', 'Customer Service', 'General Inquiry',
       'Human Resources', 'IT Support', 'Product Support',
       'Returns and Exchanges', 'Sales and Pre-Sales',
       'Service Outages and Maintenance', 'Technical Support'],
      dtype=object), array(['Change', 'Incident', 'Problem', 'Request'], dtype=object)]


In [ ]:
from sentence_transformers import SentenceTransformer

# Same embedding dimension as your saved embeddings: 384
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("✅ Embedding model loaded")
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())

C:\Users\hp\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2115.39it/s]


✅ Embedding model loaded
Embedding dimension: 384


C:\Users\hp\AppData\Local\Temp\ipykernel_11544\4121538930.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


In [ ]:
# Convert ticket into embedding
ticket_embedding = embedding_model.encode(
    [ticket],
    normalize_embeddings=True
).astype("float32")

# Search top 5 similar tickets
distances, indices = index.search(ticket_embedding, 5)

print("Top 5 similar ticket indices:")
print(indices[0])

print("\nSimilarity scores:")
print(distances[0])

Top 5 similar ticket indices:
[  414  3805   995  4579 14019]

Similarity scores:
[0.5887588  0.5839709  0.52452886 0.5141904  0.5061445 ]


In [ ]:
import pandas as pd

retrieval_df = pd.read_csv(
    "../dataset/processed/cleaned_tickets.csv"
)

print("Dataset shape:", retrieval_df.shape)

# Show retrieved tickets
for rank, idx in enumerate(indices[0], start=1):
    print(f"\n--- Result {rank} ---")
    print("Index:", idx)
    print("Type:", retrieval_df.iloc[idx]["type"])
    print("Queue:", retrieval_df.iloc[idx]["queue"])
    print("Priority:", retrieval_df.iloc[idx]["priority"])
    print("Answer:", retrieval_df.iloc[idx]["answer"])

Dataset shape: (20000, 17)

--- Result 1 ---
Index: 414
Type: Problem
Queue: Product Support
Priority: medium
Answer: I will look into your Wi-Fi problem. Could you please give more details about the issues you are facing with your Wi-Fi connectivity?

--- Result 2 ---
Index: 3805
Type: Incident
Queue: IT Support
Priority: high
Answer: I have noted your email concerning the laptop network problem. For a more thorough investigation, could you provide information on any recent software updates or configuration changes? If necessary, I can arrange a call to discuss this further and assist you in resolving the issue.

--- Result 3 ---
Index: 995
Type: Incident
Queue: Technical Support
Priority: medium
Answer: Investigate the network connection problem. For better assistance, please provide details on the router model and the number of devices connecting. Call us at <tel_num> at your convenience to discuss further steps and address the issue, which may include improving the bandwidth.

--- 

In [ ]:
retrieved_context = ""

for rank, idx in enumerate(indices[0], start=1):
    row = retrieval_df.iloc[idx]

    retrieved_context += f"""
--- Historical Incident {rank} ---
Type: {row['type']}
Queue: {row['queue']}
Priority: {row['priority']}
Resolution: {row['answer']}
"""

print(retrieved_context)


--- Historical Incident 1 ---
Type: Problem
Queue: Product Support
Priority: medium
Resolution: I will look into your Wi-Fi problem. Could you please give more details about the issues you are facing with your Wi-Fi connectivity?

--- Historical Incident 2 ---
Type: Incident
Queue: IT Support
Priority: high
Resolution: I have noted your email concerning the laptop network problem. For a more thorough investigation, could you provide information on any recent software updates or configuration changes? If necessary, I can arrange a call to discuss this further and assist you in resolving the issue.

--- Historical Incident 3 ---
Type: Incident
Queue: Technical Support
Priority: medium
Resolution: Investigate the network connection problem. For better assistance, please provide details on the router model and the number of devices connecting. Call us at <tel_num> at your convenience to discuss further steps and address the issue, which may include improving the bandwidth.

--- Historical

In [ ]:
import requests

response = requests.get("http://localhost:11434/api/tags")

print(response.status_code)
print(response.json())

200
{'models': [{'name': 'llama3.2:3b', 'model': 'llama3.2:3b', 'modified_at': '2026-08-12T22:52:31.3943523+05:30', 'size': 2019393189, 'digest': 'a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '3.2B', 'quantization_level': 'Q4_K_M', 'context_length': 131072, 'embedding_length': 3072}, 'capabilities': ['completion', 'tools']}]}


In [ ]:
prompt = f"""
You are an IT helpdesk assistant.

Analyze the following support ticket using ONLY the historical evidence provided.

TICKET:
{ticket}

HISTORICAL EVIDENCE:
{retrieved_context}

Instructions:
1. Briefly identify the likely issue.
2. Give practical troubleshooting steps.
3. Do not invent facts that are not supported by the evidence.
4. If the evidence is insufficient, clearly say so.
5. Keep the response concise.
"""

payload = {
    "model": "llama3.2:3b",
    "prompt": prompt,
    "stream": False
}

response = requests.post(
    "http://localhost:11434/api/generate",
    json=payload
)

result = response.json()

print(result["response"])

**Analysis of Support Ticket**

Based on the historical evidence provided, it appears that the user's laptop is experiencing issues with connecting to the office Wi-Fi.

**Likely Issue:**
The likely issue seems to be a Wi-Fi connectivity problem, possibly related to a configuration or firmware issue, given the resolution of Historical Incident 4 (where restarting and updating the FIRMWARE resolved the issue).

**Practical Troubleshooting Steps:**

1. **Restart laptop and wait for the network connection to re-establish**: This simple step may resolve the issue, as suggested in Historical Incident 1.
2. **Check Wi-Fi settings**: Ensure that Wi-Fi is enabled, and check the network name (SSID) and password are correct.
3. **Update firmware and drivers**: Check for any available firmware updates or driver updates related to the laptop's Wi-Fi adapter.

**Next Steps:**
If these troubleshooting steps do not resolve the issue, it would be necessary to have a more in-depth conversation with the

In [ ]:
import joblib

# Load Queue Model
queue_model = joblib.load("../models/queue/queue_model.pkl")
queue_tfidf = joblib.load("../models/queue/tfidf_vectorizer.pkl")

print("✅ Queue model loaded")

✅ Queue model loaded


In [ ]:
from scipy.sparse import hstack

# Test ticket
ticket = """
My laptop cannot connect to the office Wi-Fi.
I have tried restarting the laptop, but the problem still exists.
"""

# -----------------------------
# 1. Category / Type
# -----------------------------
category_vector = category_tfidf.transform([ticket])
predicted_type = category_model.predict(category_vector)[0]

# -----------------------------
# 2. Queue
# -----------------------------
queue_vector = queue_tfidf.transform([ticket])
predicted_queue = queue_model.predict(queue_vector)[0]

# -----------------------------
# 3. Priority
# -----------------------------
priority_vector = priority_tfidf.transform([ticket])

# Priority model was trained with:
# TF-IDF + queue + type
priority_metadata = category_encoder.transform(
    [[predicted_queue, predicted_type]]
)

priority_input = hstack([
    priority_vector,
    priority_metadata
]).tocsr()

predicted_priority = priority_model.predict(
    priority_input
)[0]

# -----------------------------
# Result
# -----------------------------
print("Ticket:")
print(ticket)

print("Category / Type:", predicted_type)
print("Queue:", predicted_queue)
print("Priority:", predicted_priority)

Ticket:

My laptop cannot connect to the office Wi-Fi.
I have tried restarting the laptop, but the problem still exists.

Category / Type: Incident
Queue: Product Support
Priority: high


C:\Users\hp\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [ ]:
import pandas as pd
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

index = faiss.read_index("../models/embeddings/ticket_index.faiss")
ticket_embeddings = np.load("../models/embeddings/ticket_embeddings.npy")

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

retrieval_df = pd.read_csv(
    "../dataset/processed/cleaned_tickets.csv"
)

print("✅ FAISS components loaded")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1420.29it/s]


✅ FAISS components loaded


In [ ]:
ticket_embedding = embedding_model.encode(
    [ticket],
    normalize_embeddings=True
).astype("float32")

distances, indices = index.search(
    ticket_embedding,
    5
)

print("Top 5 similar ticket indices:")
print(indices[0])

print("\nSimilarity scores:")
print(distances[0])

Top 5 similar ticket indices:
[  414  3805   995  4579 14019]

Similarity scores:
[0.5887588  0.5839709  0.52452886 0.5141904  0.5061445 ]


In [ ]:
retrieved_context = ""

for rank, idx in enumerate(indices[0], start=1):
    row = retrieval_df.iloc[idx]

    retrieved_context += f"""
--- Historical Incident {rank} ---
Type: {row['type']}
Queue: {row['queue']}
Priority: {row['priority']}
Resolution: {row['answer']}
"""

print(retrieved_context)


--- Historical Incident 1 ---
Type: Problem
Queue: Product Support
Priority: medium
Resolution: I will look into your Wi-Fi problem. Could you please give more details about the issues you are facing with your Wi-Fi connectivity?

--- Historical Incident 2 ---
Type: Incident
Queue: IT Support
Priority: high
Resolution: I have noted your email concerning the laptop network problem. For a more thorough investigation, could you provide information on any recent software updates or configuration changes? If necessary, I can arrange a call to discuss this further and assist you in resolving the issue.

--- Historical Incident 3 ---
Type: Incident
Queue: Technical Support
Priority: medium
Resolution: Investigate the network connection problem. For better assistance, please provide details on the router model and the number of devices connecting. Call us at <tel_num> at your convenience to discuss further steps and address the issue, which may include improving the bandwidth.

--- Historical

In [ ]:
import requests

prompt = f"""
You are an IT helpdesk assistant.

Ticket:
{ticket}

Evidence:
{retrieved_context}

Give ONLY:
1. Likely issue
2. 2-3 troubleshooting steps
3. When to escalate

Keep the answer under 100 words.
"""

payload = {
    "model": "llama3.2:3b",
    "prompt": prompt,
    "stream": False,
    "options": {
        "temperature": 0.2,
        "num_predict": 100
    }
}

response = requests.post(
    "http://localhost:11434/api/generate",
    json=payload,
    timeout=300
)

response.raise_for_status()

llm_response = response.json()["response"]

print(llm_response)

Likely Issue: The laptop is not connecting to the office Wi-Fi due to a configuration or firmware issue.

Troubleshooting Steps:

1. Check if the Wi-Fi is enabled and set to obtain an IP address automatically.
2. Restart the router and laptop, then try connecting again.
3. Run a network reset on the laptop by going to Settings > Network & Internet > Status > Reset network settings.

Escalate when: The issue persists after trying these steps, or if you're experiencing


In [ ]:
def analyze_ticket(ticket, top_k=5):

    # ==========================================
    # 1. CATEGORY / TYPE
    # ==========================================
    category_vector = category_tfidf.transform([ticket])
    predicted_type = category_model.predict(category_vector)[0]

    # ==========================================
    # 2. QUEUE
    # ==========================================
    queue_vector = queue_tfidf.transform([ticket])
    predicted_queue = queue_model.predict(queue_vector)[0]

    # ==========================================
    # 3. PRIORITY
    # ==========================================
    priority_vector = priority_tfidf.transform([ticket])

    # Priority model expects:
    # 30,000 TF-IDF + 10 queue + 4 type
    priority_metadata = category_encoder.transform(
        [[predicted_queue, predicted_type]]
    )

    priority_input = hstack([
        priority_vector,
        priority_metadata
    ]).tocsr()

    predicted_priority = priority_model.predict(
        priority_input
    )[0]

    # ==========================================
    # 4. FAISS RETRIEVAL
    # ==========================================
    ticket_embedding = embedding_model.encode(
        [ticket],
        normalize_embeddings=True
    ).astype("float32")

    distances, indices = index.search(
        ticket_embedding,
        top_k
    )

    # ==========================================
    # 5. BUILD RAG CONTEXT
    # ==========================================
    retrieved_context = ""
    retrieved_incidents = []

    for rank, idx in enumerate(indices[0], start=1):

        row = retrieval_df.iloc[idx]

        retrieved_incidents.append({
            "rank": rank,
            "index": int(idx),
            "similarity": float(distances[0][rank - 1]),
            "type": row["type"],
            "queue": row["queue"],
            "priority": row["priority"],
            "answer": row["answer"]
        })

        retrieved_context += f"""
Historical Incident {rank}:
Type: {row['type']}
Queue: {row['queue']}
Priority: {row['priority']}
Resolution: {row['answer']}
"""

    # ==========================================
    # 6. OLLAMA / RAG
    # ==========================================
    prompt = f"""
You are an IT helpdesk assistant.

Ticket:
{ticket}

Predicted Type: {predicted_type}
Predicted Queue: {predicted_queue}
Predicted Priority: {predicted_priority}

Historical Evidence:
{retrieved_context}

Give ONLY:
1. Likely issue
2. 2-3 troubleshooting steps
3. When to escalate

Use the historical evidence.
Do not invent unsupported facts.
Keep the answer under 100 words.
"""

    payload = {
        "model": "llama3.2:3b",
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.2,
            "num_predict": 100
        }
    }

    response = requests.post(
        "http://localhost:11434/api/generate",
        json=payload,
        timeout=300
    )

    response.raise_for_status()

    resolution = response.json()["response"]

    # ==========================================
    # 7. FINAL RESULT
    # ==========================================
    return {
        "ticket": ticket,
        "category": predicted_type,
        "queue": predicted_queue,
        "priority": predicted_priority,
        "retrieved_incidents": retrieved_incidents,
        "resolution": resolution
    }

In [ ]:
result = analyze_ticket(
    "My laptop cannot connect to the office Wi-Fi. "
    "I have restarted it but the problem still exists."
)

print("================================")
print("       AI TICKET ANALYSIS")
print("================================")

print("\nCategory:", result["category"])
print("Queue:", result["queue"])
print("Priority:", result["priority"])

print("\nSimilar Incidents:")

for incident in result["retrieved_incidents"]:
    print(
        f"#{incident['rank']} "
        f"| Similarity: {incident['similarity']:.3f} "
        f"| Queue: {incident['queue']} "
        f"| Priority: {incident['priority']}"
    )

print("\nResolution:")
print(result["resolution"])

C:\Users\hp\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


       AI TICKET ANALYSIS

Category: Incident
Queue: Product Support
Priority: low

Similar Incidents:
#1 | Similarity: 0.624 | Queue: Product Support | Priority: medium
#2 | Similarity: 0.600 | Queue: IT Support | Priority: high
#3 | Similarity: 0.538 | Queue: Technical Support | Priority: medium
#4 | Similarity: 0.534 | Queue: Technical Support | Priority: high
#5 | Similarity: 0.525 | Queue: IT Support | Priority: high

Resolution:
Likely Issue: The laptop is unable to connect to the office Wi-Fi due to a configuration or firmware issue.

Troubleshooting Steps:

1. Check if there are any recent software updates or configuration changes that may be causing the issue.
2. Restart the router and laptop, then try connecting again.
3. Ensure that the Wi-Fi settings on the laptop are set to use the correct network name and password.

Escalate: If none of these steps resolve the issue, escalate to IT Support
